# Stokastisk 0.1.0 — Qwen2.5 inference on Kaggle 2× Tesla T4

This notebook installs the integrated Stokastisk/vLLM 0.25.1 wheel directly from the Hugging Face CDN, installs the release's 19 mandatory Python runtime dependencies, validates the Kaggle GPU environment, and runs Qwen2.5-0.5B-Instruct through the Stokastisk SDK with the **TRITON_ATTN** backend.

**Notebook design:** ordinary Kaggle cells only. No `main()` function, no `%%writefile`, no generated `.py` file, and no triple-quoted Python source passed to a subprocess.

**GPU choice:** Qwen2.5-0.5B-Instruct comfortably fits on one T4, so the recommended default is tensor parallel size 1. Both T4s are detected; the second remains available for another workload. An optional two-GPU configuration is shown near the end.

## 1. Confirm the Kaggle runtime

Enable **GPU** in the Kaggle notebook settings before running this cell. This release targets Python 3.13, CUDA 13.0, PyTorch 2.11.0+cu130, Triton 3.6.0, and Tesla T4 / SM75.

In [ ]:
import sys
import torch
import triton

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('Triton:', triton.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU count:', torch.cuda.device_count())

assert sys.version_info[:2] == (3, 13), 'This wheel targets Python 3.13.'
assert torch.__version__.startswith('2.11.0+cu130'), 'Unexpected PyTorch build; do not replace Kaggle CUDA packages blindly.'
assert torch.version.cuda == '13.0', 'This release targets CUDA 13.0.'
assert triton.__version__ == '3.6.0', 'This release targets Triton 3.6.0.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
assert torch.cuda.device_count() >= 2, 'This notebook expects Kaggle 2× Tesla T4 GPUs.'

for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    print(f'GPU {gpu_id}: {props.name}, compute capability {props.major}.{props.minor}, {props.total_memory / 2**30:.2f} GiB')
    assert props.name == 'Tesla T4', f'Expected Tesla T4, found {props.name}'
    assert (props.major, props.minor) == (7, 5), 'Expected T4 compute capability SM75.'

## 2. Install the integrated Stokastisk 0.1.0 / vLLM 0.25.1 wheel from Github

In [ ]:
!pip install stokastisk --extra-index-url https://stokastisk-ai.github.io/stokastisk/simple/

## 3. Validate the installed SDK and the T4 backend policy

The integrated wheel contains the `stokastisk` SDK and the bundled vLLM 0.25.1 source/native components. This cell checks package versions, GPU visibility, and that the release's prohibited Flash-family backend report is clean.

In [ ]:
import importlib.metadata as metadata
import json
import stokastisk
import torch
import triton

print('Stokastisk:', stokastisk.__version__)
print('Bundled vLLM distribution:', metadata.version('vllm'))
print('PyTorch:', torch.__version__)
print('Triton:', triton.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())

assert stokastisk.__version__ == '0.1.0'
assert metadata.version('vllm') == '0.25.1'
assert torch.cuda.device_count() == 2

policy_report = stokastisk.prohibited_report()
print(json.dumps(policy_report, indent=2))
assert policy_report.get('clean') is True, 'A prohibited Flash-family component was detected.'

print('\nQwen2.5 model compatibility probe:')
print(json.dumps(stokastisk.check_model('Qwen/Qwen2.5-0.5B-Instruct'), indent=2, default=str))

## 4. Run Qwen2.5-0.5B-Instruct inference directly in this notebook cell

No subprocess wrapper or `main()` function is used. `stokastisk.LLM` applies the release defaults: FP16, eager execution, pinned `TRITON_ATTN`, and XGrammar structured outputs. The model weights are downloaded from Hugging Face on first use.

In [ ]:
from stokastisk import LLM, StokastiskConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# This model fits on one T4; TP=1 avoids unnecessary PCIe/NCCL overhead.
config = StokastiskConfig(
    tensor_parallel_size=1,
    dtype='half',
    attention_backend='TRITON_ATTN',
    max_model_len=2048,
    gpu_memory_utilization=0.88,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)

with LLM(MODEL_ID, config=config) as llm:
    active_backend = llm.verify_attention_backend()
    print('Verified attention backend:', active_backend)
    assert active_backend == 'TRITON_ATTN'

    completion = llm.complete(
        'Explain in three short sentences why the sky appears blue.',
        max_tokens=96,
        temperature=0.2,
    )
    print('\n=== Completion ===\n', completion)

    chat_result = llm.chat([
        {'role': 'system', 'content': 'You are a concise assistant.'},
        {'role': 'user', 'content': 'Give me five practical uses of Python for data analysis.'},
    ])
    print('\n=== Chat ===\n', chat_result[0].outputs[0].text)

    schema = {
        'type': 'object',
        'properties': {
            'model': {'type': 'string', 'maxLength': 40},
            'gpu': {'type': 'string', 'maxLength': 40},
            'purpose': {'type': 'string', 'maxLength': 120},
        },
        'required': ['model', 'gpu', 'purpose'],
        'additionalProperties': False,
    }
    structured_result = llm.generate_structured(
        ['Return a short JSON object describing this Qwen2.5 inference test on a Tesla T4.'],
        schema,
        max_tokens=96,
        temperature=0.0,
    )
    print('\n=== Schema-constrained JSON ===\n', structured_result)

print('\nThe Stokastisk engine context has closed.')

## 5. Inspect GPU memory after inference


In [ ]:
for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    print({
        'device': gpu_id,
        'name': props.name,
        'compute_capability': f'{props.major}.{props.minor}',
        'total_memory_GiB': round(total_bytes / 2**30, 2),
        'free_memory_GiB': round(free_bytes / 2**30, 2),
    })

## 6. Optional: distribute a larger model across both T4 GPUs

Do not run this cell for the 0.5B model unless you specifically want to test tensor parallelism. For a larger compatible model that needs both GPUs, change the config to `tensor_parallel_size=2`. Two T4s communicate over PCIe (not NVLink), so TP=2 can be slower than a single GPU when the model already fits on one card.

```python
config_2gpu = StokastiskConfig(
    tensor_parallel_size=2,
    dtype='half',
    attention_backend='TRITON_ATTN',
    max_model_len=2048,
    gpu_memory_utilization=0.88,
)
with LLM('Qwen/Qwen2.5-3B-Instruct', config=config_2gpu) as llm:
    print(llm.verify_attention_backend())
    print(llm.complete('Explain tensor parallelism briefly.', max_tokens=80))
```

The 3B example is illustrative, not a claim that every Qwen2.5 checkpoint or context length has been verified by this exact release.

In [ ]:

# ============================================================
# STEP 6: TWO-GPU TENSOR-PARALLEL INFERENCE
# Stokastisk 0.1.0 | vLLM 0.25.1
# Python 3.13 | CUDA 13.0 | 2x NVIDIA Tesla T4 (SM75)
# ============================================================

import sys
import json
import torch
import stokastisk

from stokastisk import LLM, StokastiskConfig

# ------------------------------------------------------------
# 1. Runtime validation
# ------------------------------------------------------------

assert sys.version_info[:2] == (3, 13), "Python 3.13 required."
assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.device_count() == 2, "Exactly two visible GPUs required."
assert stokastisk.__version__ == "0.1.0"

print("Stokastisk:", stokastisk.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

for gpu_id in range(2):
    props = torch.cuda.get_device_properties(gpu_id)

    print(
        f"GPU {gpu_id}: {props.name} | "
        f"SM {props.major}.{props.minor} | "
        f"{props.total_memory / 2**30:.2f} GiB"
    )

    assert props.name == "Tesla T4"
    assert (props.major, props.minor) == (7, 5)

# ------------------------------------------------------------
# 2. Select a larger Qwen2.5 model
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

compatibility = stokastisk.check_model(MODEL_ID)

print("\nModel compatibility:")
print(json.dumps(compatibility, indent=2, default=str))

# ------------------------------------------------------------
# 3. Configure tensor parallelism across both GPUs
# ------------------------------------------------------------

config_2gpu = StokastiskConfig(
    tensor_parallel_size=2,
    pipeline_parallel_size=1,

    dtype="half",

    attention_backend="TRITON_ATTN",

    max_model_len=2048,

    gpu_memory_utilization=0.88,

    enforce_eager=True,

    disable_custom_all_reduce=True,
)

print("\nTensor-parallel configuration:")
print(json.dumps(config_2gpu.as_dict(), indent=2, default=str))

# ------------------------------------------------------------
# 4. Launch the Stokastisk inference engine
# ------------------------------------------------------------

with LLM(MODEL_ID, config=config_2gpu) as llm:

    active_backend = llm.verify_attention_backend()

    print("\nVerified attention backend:", active_backend)

    assert active_backend == "TRITON_ATTN"

    # Completion
    completion = llm.complete(
        "Explain how tensor parallelism distributes "
        "a neural network across two GPUs.",
        max_tokens=128,
        temperature=0.2,
    )

    print("\n=== TWO-GPU COMPLETION ===")
    print(completion)

    # Chat
    chat_result = llm.chat([
        {
            "role": "system",
            "content": "You are a concise technical assistant."
        },
        {
            "role": "user",
            "content": "Explain the difference between "
                       "single-GPU inference and TP=2 inference."
        },
    ])

    print("\n=== TWO-GPU CHAT ===")
    print(chat_result[0].outputs[0].text)

    # --------------------------------------------------------
    # 5. Inspect GPU memory while the engine is active
    # --------------------------------------------------------

    print("\n=== GPU MEMORY DURING INFERENCE ===")

    for gpu_id in range(2):

        torch.cuda.synchronize(gpu_id)

        allocated = torch.cuda.memory_allocated(gpu_id)
        reserved = torch.cuda.memory_reserved(gpu_id)
        free, total = torch.cuda.mem_get_info(gpu_id)

        print({
            "gpu": gpu_id,
            "allocated_GiB": round(allocated / 2**30, 2),
            "reserved_GiB": round(reserved / 2**30, 2),
            "free_GiB": round(free / 2**30, 2),
            "total_GiB": round(total / 2**30, 2),
        })

print("\nTwo-GPU Stokastisk inference engine closed.")

# ------------------------------------------------------------
# 6. Confirm GPU memory after engine shutdown
# ------------------------------------------------------------

print("\n=== GPU MEMORY AFTER SHUTDOWN ===")

for gpu_id in range(2):

    free, total = torch.cuda.mem_get_info(gpu_id)

    print({
        "gpu": gpu_id,
        "free_GiB": round(free / 2**30, 2),
        "total_GiB": round(total / 2**30, 2),
    })

print("\nSTEP 6 COMPLETED.")


## Release and installation notes

- Stokastisk repository: https://github.com/stokastisk-ai/stokastisk
- Stokastisk v0.1.0 release: https://github.com/stokastisk-ai/stokastisk/releases/tag/v0.1.0
- Hugging Face repository: https://huggingface.co/waqasm86/stokastisk
- Direct wheel CDN URL: https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-3-cp313-cp313-linux_x86_64.whl

Run the notebook from top to bottom in a fresh Kaggle session (GPU T4 × 2, Internet ON). Verified on Kaggle 2 × T4: see docs/KAGGLE_ACCEPTANCE.md.


## 7. Pipeline parallelism across both NVIDIA Tesla T4 GPUs

This step tests pipeline parallelism (PP=2) using Stokastisk 0.1.0 and the bundled vLLM 0.25.1 runtime.

Unlike Step 6, which used tensor parallelism (TP=2), this step assigns sequential model layers to two pipeline stages:

- GPU 0: pipeline stage 0
- GPU 1: pipeline stage 1

### Configuration

- Model: `Qwen/Qwen2.5-3B-Instruct`
- Tensor parallel size: `1`
- Pipeline parallel size: `2`
- Precision: FP16 (`half`)
- Attention backend: `TRITON_ATTN`
- Maximum model length: `2048`
- GPU memory utilization: `0.88`
- Eager execution: enabled
- Custom all-reduce: disabled

The test uses the same model and inference configuration as Step 6, changing only the parallelism strategy. This makes the two tests easier to compare.

### Validation criteria

1. Python 3.13 and Stokastisk 0.1.0 are available.
2. Both GPUs are Tesla T4 devices with SM 7.5.
3. The configuration reports TP=1 and PP=2.
4. vLLM initializes with `tensor_parallel_size=1` and `pipeline_parallel_size=2`.
5. Both pipeline stages initialize successfully.
6. The attention backend is verified as `TRITON_ATTN`.
7. Completion and chat inference return generated text.
8. Both GPUs release their allocated memory after engine shutdown.

**Important:** This is a pipeline-parallel execution test, not a performance benchmark. Qwen2.5-3B-Instruct can fit on a single T4, so this model is used to compare the parallel execution modes rather than demonstrate the maximum model capacity of two GPUs.


In [ ]:

# ============================================================
# STEP 7: PIPELINE PARALLELISM (PP=2)
# Stokastisk 0.1.0 | vLLM 0.25.1
# Python 3.13 | CUDA 13.0 | PyTorch 2.11.0
# Kaggle: 2x NVIDIA Tesla T4 (SM75)
# ============================================================

import json
import torch
import stokastisk

from stokastisk import LLM, StokastiskConfig


# ------------------------------------------------------------
# 1. Validate the Kaggle runtime
# ------------------------------------------------------------

print("=" * 65)
print("STEP 7: STOKASTISK PIPELINE PARALLELISM")
print("=" * 65)

print("Stokastisk:", stokastisk.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

assert stokastisk.__version__ == "0.1.0"
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

assert torch.__version__.startswith("2.11.0")
assert torch.version.cuda == "13.0"

for gpu_id in range(2):

    props = torch.cuda.get_device_properties(gpu_id)

    print(
        f"GPU {gpu_id}: {props.name} | "
        f"SM {props.major}.{props.minor} | "
        f"{props.total_memory / 2**30:.2f} GiB"
    )

    assert props.name == "Tesla T4"
    assert (props.major, props.minor) == (7, 5)


# ------------------------------------------------------------
# 2. Select the same model used in Step 6
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

compatibility = stokastisk.check_model(
    MODEL_ID,
    architectures=["Qwen2ForCausalLM"],
)

print("\nModel compatibility probe:")
print(json.dumps(compatibility, indent=2, default=str))


# ------------------------------------------------------------
# 3. Configure pipeline parallelism
# ------------------------------------------------------------

config_pp2 = StokastiskConfig(

    # One tensor-parallel partition.
    tensor_parallel_size=1,

    # Divide sequential model layers across two GPUs.
    pipeline_parallel_size=2,

    dtype="half",

    attention_backend="TRITON_ATTN",

    max_model_len=2048,

    gpu_memory_utilization=0.88,

    enforce_eager=True,

    disable_custom_all_reduce=True,
)


# ------------------------------------------------------------
# 4. Validate the configuration before engine initialization
# ------------------------------------------------------------

config_report = config_pp2.as_dict()

print("\nPipeline-parallel configuration:")
print(json.dumps(config_report, indent=2, default=str))

assert config_report["tensor_parallel_size"] == 1
assert config_report["pipeline_parallel_size"] == 2
assert config_report["attention_backend"] == "TRITON_ATTN"
assert config_report["dtype"] == "half"

print("\nTP=1 and PP=2 configuration validated.")


# ------------------------------------------------------------
# 5. Launch the pipeline-parallel inference engine
# ------------------------------------------------------------

with LLM(MODEL_ID, config=config_pp2) as llm:

    # Verify the attention backend selected by vLLM.
    active_backend = llm.verify_attention_backend()

    print("\nVerified attention backend:", active_backend)

    assert active_backend == "TRITON_ATTN"

    # --------------------------------------------------------
    # 5A. Completion inference
    # --------------------------------------------------------

    completion = llm.complete(

        "Explain pipeline parallelism in large language "
        "models. Describe how two NVIDIA GPUs process "
        "different sequential layers of the same model.",

        max_tokens=128,

        temperature=0.2,
    )

    print("\n" + "=" * 65)
    print("PIPELINE PARALLEL COMPLETION")
    print("=" * 65)

    print(completion)


    # --------------------------------------------------------
    # 5B. Chat inference
    # --------------------------------------------------------

    chat_result = llm.chat([

        {
            "role": "system",
            "content": "You are a concise technical assistant."
        },

        {
            "role": "user",
            "content": (
                "Compare tensor parallelism and pipeline "
                "parallelism when running an LLM on two GPUs."
            )
        },

    ])

    print("\n" + "=" * 65)
    print("PIPELINE PARALLEL CHAT")
    print("=" * 65)

    print(chat_result[0].outputs[0].text)


    # --------------------------------------------------------
    # 5C. Inspect GPU memory while the engine is active
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("GPU MEMORY DURING PIPELINE INFERENCE")
    print("=" * 65)

    for gpu_id in range(2):

        torch.cuda.synchronize(gpu_id)

        free, total = torch.cuda.mem_get_info(gpu_id)

        allocated = torch.cuda.memory_allocated(gpu_id)
        reserved = torch.cuda.memory_reserved(gpu_id)

        print({

            "gpu": gpu_id,

            "name": torch.cuda.get_device_name(gpu_id),

            "allocated_GiB": round(
                allocated / 2**30, 2
            ),

            "reserved_GiB": round(
                reserved / 2**30, 2
            ),

            "free_GiB": round(
                free / 2**30, 2
            ),

            "total_GiB": round(
                total / 2**30, 2
            ),

        })


# ------------------------------------------------------------
# 6. Confirm engine shutdown and GPU memory recovery
# ------------------------------------------------------------

print("\nPipeline-parallel Stokastisk engine closed.")

print("\n" + "=" * 65)
print("GPU MEMORY AFTER ENGINE SHUTDOWN")
print("=" * 65)

for gpu_id in range(2):

    free, total = torch.cuda.mem_get_info(gpu_id)

    print({

        "gpu": gpu_id,

        "free_GiB": round(
            free / 2**30, 2
        ),

        "total_GiB": round(
            total / 2**30, 2
        ),

    })


# ------------------------------------------------------------
# 7. Final Step 7 result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 7 PIPELINE PARALLELISM TEST COMPLETED")
print("=" * 65)

print("Model:", MODEL_ID)
print("Tensor parallel size:", config_report["tensor_parallel_size"])
print("Pipeline parallel size:", config_report["pipeline_parallel_size"])
print("Attention backend:", active_backend)
print("GPU count:", torch.cuda.device_count())

print("\nExpected execution:")
print("GPU 0: Pipeline stage 0")
print("GPU 1: Pipeline stage 1")

print("\nSTEP 7 COMPLETED.")
